# Lab 03: Inventory System Simulation (Continuous Review)

**Problem Statement:**
Build a simulation for an inventory system where demand is normally distributed. Implement a reorder point (s) and reorder quantity (S) policy. Track total holding costs and shortage costs over a fiscal year.


### Theory and Formulas

**Continuous Review (s, S) Policy**:
- **$s$**: Reorder point. When the inventory position drops to or below $s$, an order is placed.
- **$S$**: Order-up-to level (or reorder quantity policy). We order an amount $Q = S - I$ to bring the inventory position up to $S$.
- **Inventory Position**: On-hand inventory + On-order inventory - Backorders.

**Parameters for this simulation**:
- **Demand**: Normally distributed $N(\mu, \sigma^2)$ per day.
- **Lead Time**: Time until an order arrives. (We will use a fixed lead time of 3 days).
- **Time Horizon**: One fiscal year (365 days).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Parameters ---
FISCAL_YEAR = 365
MEAN_DEMAND = 20.0
STD_DEMAND = 5.0

s_REORDER_POINT = 40     # s
S_ORDER_UP_TO = 100      # S
LEAD_TIME = 3            # Days until order arrives

HOLDING_COST = 0.5       # Per unit per day
SHORTAGE_COST = 5.0      # Per unit short

# --- State Variables ---
on_hand_inventory = S_ORDER_UP_TO
on_order_inventory = 0
days_until_order_arrives = 0
order_quantity_pending = 0

total_holding_cost = 0.0
total_shortage_cost = 0.0

# --- Tracking ---
history_inventory = []
history_days = list(range(1, FISCAL_YEAR + 1))

print("--- Continuous Review (s, S) Simulation ---")

for day in history_days:
    # 1. Receive pending orders
    if days_until_order_arrives == 0 and order_quantity_pending > 0:
        on_hand_inventory += order_quantity_pending
        on_order_inventory -= order_quantity_pending
        order_quantity_pending = 0
    elif order_quantity_pending > 0:
        days_until_order_arrives -= 1
        
    # 2. Daily Demand (Normally distributed)
    # Ensure demand is at least 0
    demand = max(0, int(np.round(np.random.normal(MEAN_DEMAND, STD_DEMAND))))
    
    # 3. Fulfill demand
    if on_hand_inventory >= demand:
        on_hand_inventory -= demand
        total_holding_cost += on_hand_inventory * HOLDING_COST
    else:
        shortage = demand - on_hand_inventory
        total_shortage_cost += shortage * SHORTAGE_COST
        on_hand_inventory = 0
        
    history_inventory.append(on_hand_inventory)
    
    # 4. End of day review
    inventory_position = on_hand_inventory + on_order_inventory
    if inventory_position <= s_REORDER_POINT and order_quantity_pending == 0:
        # Place an order
        order_quantity_pending = S_ORDER_UP_TO - inventory_position
        on_order_inventory += order_quantity_pending
        days_until_order_arrives = LEAD_TIME
        
print(f"Simulation ended after {FISCAL_YEAR} days.")
print(f"Total Holding Cost : ${total_holding_cost:.2f}")
print(f"Total Shortage Cost: ${total_shortage_cost:.2f}")
print(f"Total Costs        : ${total_holding_cost + total_shortage_cost:.2f}")


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(history_days, history_inventory, color='#9467bd', linewidth=1.5)
plt.axhline(S_ORDER_UP_TO, color='gray', linestyle='--', label='Order-up-to Level (S)')
plt.axhline(s_REORDER_POINT, color='red', linestyle='--', label='Reorder Point (s)')
plt.title("Inventory Level Over a Fiscal Year - (s, S) Policy", fontsize=14, fontweight='bold')
plt.xlabel("Day of Year", fontsize=12)
plt.ylabel("On-Hand Inventory", fontsize=12)
plt.legend()
plt.grid(alpha=0.4)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
